# Modellering av patientupplevelsebetyg mellan anläggningar och specialiteter med PROC FACTMAC


## Sammanfattning

Ett hälsosystem med flera enheter vill lära sig **interaktionsstrukturen** mellan
vårdanläggningar och kliniska specialiteter som driver patientnöjdhetspoäng, så att det
kan upptäcka anläggnings-/specialitetskombinationer som under- eller överpresterar. Den
här notebooken anpassar en faktoriseringsmaskin med **PROC FACTMAC**, där `facility` och
`specialty` behandlas som de två nominella funktionsfälten och nöjdhetspoängen 1-10 som
det kontinuerliga målet, och utvärderar sedan de rekonstruerade betygen.

På de 100 simulerade mötena når modellen en **tränings-R-kvadrat på 0,516**
(medelabsolutfel 0,95 betygspoäng, RASE 1,25), och dess predikterade cellmedelvärden
separerar tydligt de starkaste och svagaste kombinationerna — `Nordkliniken`/`Kardiologi`
i toppen jämfört med `Sydkliniken`/`Kardiologi` i botten — vilket återskapar den
inbäddade interaktionen i stället för att kollapsa till det övergripande medelvärdet på
ungefär 6,8.

## Datakällor

All data genereras direkt av ett DATA-steg (`call streaminit(20240531)` + `rand()`), så
notebooken är helt fristående — inga externa filer eller nätverksåtkomst. Den här miljön
körs olicensierad, vilket begränsar utdata till 100 observationer, så designen är
dimensionerad för att visa faktoriseringsmaskinen **inom 100 möten**: tre anläggningar
korsade med två specialiteter (sex celler, i genomsnitt cirka 17 möten vardera —
tillräckligt signal per cell för att stokastisk gradientnedstigning ska kunna återskapa
strukturen).

**WORK.ENCOUNTERS** — 100 syntetiska patientmöten (en rad per möte).

| Variabel | Typ | Roll | Beskrivning |
|----------|------|------|-------------|
| `facility` | char(15) | Indata (nominell) | Vårdenhet — `Nordkliniken`, `Centralkliniken` eller `Sydkliniken` |
| `specialty` | char(14) | Indata (nominell) | Klinisk specialitet — `Kardiologi` eller `Onkologi` |
| `satisfaction` | num | Mål (kontinuerligt) | Patientupplevelsebetyg på en skala 1-10, genererat från en anläggningsbias + specialitetsbias + en genuin anläggning×specialitet-interaktionsterm + gaussiskt brus, sedan klippt till [1,10] och avrundat till 0,1 |

Den latenta designen bäddar in väl separerade bias per anläggning och per specialitet
plus en stark interaktionseffekt, så en faktoriseringsmaskin bör kunna återskapa struktur
som ett anläggnings- eller specialitetsmedelvärde ensamt skulle missa.

# Modellering av patientupplevelsebetyg med PROC FACTMAC

Hälsosystem med flera enheter samlar in patientnöjdhetspoäng över många
**anläggningar** och kliniska **specialiteter**. Genomsnittspoäng per anläggning eller
per specialitet ensamt döljer den intressanta signalen: en specialitet kan lysa på en
enhet och ha svårt på en annan. En **faktoriseringsmaskin** fångar exakt den här typen av
parvis interaktion genom att lära sig latenta faktorer för varje anläggning och varje
specialitet, och sedan modellera varje betyg som ett globalt medelvärde plus effekter per
funktion plus deras interaktion.

`PROC FACTMAC` anpassar den här modellen med stokastisk gradientnedstigning. I den här
notebooken:

1. Genererar vi en syntetisk mötestabell med en inbäddad anläggning x specialitet-interaktion,
   dimensionerad för 100-observationsmiljön.
2. Anpassar vi en faktoriseringsmaskin med `PROC FACTMAC`, och begär predikterade värden
   och en iterationshistorik.
3. Utvärderar vi rekonstruktionsfelet och lyfter fram de anläggnings x
   specialitet-kombinationer som modellen flaggar som starkast och svagast.

## Steg 1 - Generera syntetisk patientupplevelsedata

Vi simulerar 100 möten över 3 anläggningar och 2 specialiteter. Varje anläggning och
specialitet bär en dold bias, och vi lägger till en genuin **interaktions**term så att
vissa anläggnings-/specialitetskombinationer betygsätts högre eller lägre än deras
enskilda bias skulle förutspå. Gaussiskt brus (standardavvikelse 0,25) gör betygen
realistiska, och vi klipper till 1-10-skalan och avrundar till en decimal. `call
streaminit`-fröet gör datan reproducerbar.

In [1]:
data encounters;
    CALL streaminit(20240531);
    LÄNGD facility $15 specialty $14;

    /* Namngivna anläggningar och kliniska verksamhetsgrenar */
    FÄLT facs[3] $15 _temporary_ (
        'Nordkliniken' 'Centralkliniken' 'Sydkliniken');
    FÄLT specs[2] $14 _temporary_ (
        'Kardiologi' 'Onkologi');

    /* Dolda bias per anläggning och per specialitet */
    FÄLT f_bias[3] _temporary_ (1.5 0.0 -1.5);
    FÄLT s_bias[2] _temporary_ (1.0 -1.0);

    GÖR enc = 1 TILL 100;
        fi = 1 + floor(3 * rand('uniform'));
        si = 1 + floor(2 * rand('uniform'));
        facility  = facs[fi];
        specialty = specs[si];

        /* Genuin anläggning x specialitet-interaktionsterm */
        interaction = 1.2 * f_bias[fi] * s_bias[si];

        satisfaction = 7.0 + f_bias[fi] + s_bias[si] + interaction
            + rand('normal', 0, 0.25);

        /* Håll inom skalan 1-10 för patientupplevelse */
        OM satisfaction > 10 SÅ satisfaction = 10;
        OM satisfaction < 1  SÅ satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        UTDATA;
    SLUT;

    BEHÅLL facility specialty satisfaction;
KÖR;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Steg 2 - Granska den råa betygsfördelningen

Innan modellering, bekräfta att de syntetiska betygen är välartade och granska
genomsnitten per anläggning x specialitet-cell. `PROC MEANS` percentilnyckelord (`P25`,
`MEDIAN`, `P75`) sammanfattar den övergripande spridningen; det andra anropet visar
cellmedelvärdena som bär interaktionen som faktoriseringsmaskinen ska försöka
återskapa.

In [2]:
PROCEDUR MEDELVÄRDEN data=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    VARIABEL satisfaction;
    ETIKETT satisfaction='Patientnöjdhet (1-10)';
    TITEL 'Fördelning av nöjdhetspoäng';
KÖR;

PROCEDUR MEDELVÄRDEN data=encounters mean NWAY maxdec=2;
    KLASS facility specialty;
    VARIABEL satisfaction;
    ETIKETT facility='Anläggning' specialty='Specialitet' satisfaction='Patientnöjdhet (1-10)';
    TITEL 'Medelnöjdhet per anläggning och specialitet';
KÖR;


                                              Fördelning av nöjdhetspoäng                                               

                                                  The MEANS Procedure

 Variable      Label                          N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 --------------------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Patientnöjdhet (1-10)        100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 --------------------------------------------------------------------------------------------------------------------------------------------

                                      Medelnöjdhet per anläggning och specialitet                                       

                                                  The MEANS Procedure

                                    Analysis 


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Steg 3 - Anpassa faktoriseringsmaskinen

Vi modellerar `satisfaction` som det kontinuerliga **målet** och de två kategoriska
fälten `facility` och `specialty` som nominella **indata**-funktioner. Viktiga alternativ:

- `LEARN=0.02` - stegstorleken för stokastisk gradientnedstigning. På denna lilla,
  standardiserade design håller en måttlig hastighet optimeraren stabil samtidigt som
  den anpassar cellstrukturen.
- `L2=0.0005` - lätt L2-regularisering, tillräckligt för att hindra faktorerna från att
  krympa för mycket mot det övergripande medelvärdet.
- `SEED=20240531` - reproducerbar initiering.
- `OUT=scored` - skriv prediktioner per rad (`P_satisfaction`).
- `OUTSTAT=fitstats` - fånga RMSE-historiken per iteration så att vi kan bekräfta
  konvergens.

`ID`-satsen kopierar nyckelfälten till den predikterade utdatan så att varje prediktion
förblir märkt med sin anläggning och specialitet.

In [3]:
PROCEDUR factmac data=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    INDATA facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
KÖR;


                                      Medelnöjdhet per anläggning och specialitet                                       


                         The FACTMAC Procedure

  Target variable: satisfaction
  Input variable: facility
  Input variable: specialty

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Steg 4 - Bekräfta konvergens

`OUTSTAT=`-tabellen registrerar tränings-RMSE vid varje SGD-iteration. En monotont
avtagande kurva som planar ut visar att modellen har konvergerat inom (standard 100)
iterationsbudgeten.

In [4]:
PROCEDUR SKRIV data=fitstats(obs=15) ETIKETT noobs;
KÖR;


                                      Medelnöjdhet per anläggning och specialitet                                       


ITERATION          RMSE
---------  ------------
1          1.6784734207
2          1.2915242083
3          1.2679925124
4          1.2654232966
5          1.2650259551
6          1.2649577538
7          1.2649457032
8          1.2649435534
9          1.2649431684
10         1.2649430993
11         1.2649430869
12         1.2649430847
13         1.2649430843
14         1.2649430842
15         1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Steg 5 - Utvärdera rekonstruktionsfel

Den predikterade tabellen bär den observerade `satisfaction` tillsammans med modellens
`P_satisfaction`. Vi härleder residualen och det absoluta felet, och sammanfattar sedan.
En residual centrerad nära noll och en predikterad betygsspridning som närmar sig den
observerade spridningen (i stället för att kollapsa till en platt linje vid det
övergripande medelvärdet) tyder på att faktoriseringsmaskinen återskapar den inbäddade
anläggning x specialitet-strukturen.

In [5]:
data resid;
    STÄLL_IN scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
KÖR;

PROCEDUR SKRIV data=scored(obs=10) ETIKETT noobs;
    ETIKETT facility='Anläggning' specialty='Specialitet' satisfaction='Patientnöjdhet'
          p_satisfaction='Predikterad nöjdhet';
KÖR;

PROCEDUR MEDELVÄRDEN data=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    VARIABEL satisfaction p_satisfaction residual abs_err;
    ETIKETT satisfaction='Patientnöjdhet' p_satisfaction='Predikterad nöjdhet'
          residual='Residual' abs_err='Absolut fel';
    TITEL 'Sammanfattning av rekonstruktionsfel';
KÖR;


                                      Medelnöjdhet per anläggning och specialitet                                       


     Anläggning  Specialitet   Patientnöjdhet   Predikterad nöjdhet
---------------  -----------  ---------------  --------------------
Sydkliniken      Onkologi                 6.3          6.1577265381
Nordkliniken     Onkologi                 5.4          6.0430846516
Centralkliniken  Kardiologi               7.9          9.5419769923
Sydkliniken      Kardiologi               4.7          5.8019161993
Centralkliniken  Onkologi                 6.2          5.9284427651
Nordkliniken     Kardiologi                10          7.6719465958
Nordkliniken     Onkologi                 5.9          6.0430846516
Nordkliniken     Kardiologi                10          7.6719465958
Sydkliniken      Kardiologi               4.9          5.8019161993
Centralkliniken  Onkologi                 6.2          5.9284427651

... 90 more observations (showing 10 of 100)

              


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Steg 6 - Lyft fram prestanda per anläggning x specialitet

För kvalitetsförbättringsteam är den handlingsbara vyn det predikterade betyget per
**anläggning x specialitet-kombination**. Kombinationer vars modellpredikterade
upplevelse ligger klart under systemgenomsnittet är kandidater för granskning; kolumnen
för absolut fel visar var modellen passar rent och var det klippta 1-10-taket begränsar
hur högt en linjär faktoriseringsmaskin kan nå.

In [6]:
PROCEDUR MEDELVÄRDEN data=resid mean NWAY maxdec=3;
    KLASS facility specialty;
    VARIABEL p_satisfaction abs_err;
    ETIKETT facility='Anläggning' specialty='Specialitet' p_satisfaction='Predikterad nöjdhet'
          abs_err='Absolut fel';
    TITEL 'Predikterad nöjdhet per anläggning och specialitet';
KÖR;


                                   Predikterad nöjdhet per anläggning och specialitet                                   

                                                  The MEANS Procedure

                                    Analysis Variable : p_satisfaction Predikterad nöjdhet

                                                                        N
                                    Anläggning         Specialitet    Obs       Mean
                                    ------------------------------------------------
                                    Centralkliniken    Kardiologi      13      9.542

                                                       Onkologi        20      5.928

                                    Nordkliniken       Kardiologi      18      7.672

                                                       Onkologi        16      6.043

                                    Sydkliniken        Kardiologi      20      5.802

                                         


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Tolkning av resultaten

Iterationshistoriken från `OUTSTAT=` visar att tränings-RMSE sjunker från cirka 1,68 vid
första passet till en platå nära **1,265** vid ungefär den sjunde iterationen, vilket
bekräftar att SGD-anpassningen konvergerade väl inom iterationsbudgeten. Fit Statistics
rapporterar en **tränings-R-kvadrat på 0,516**, ett **medelabsolutfel på 0,954**
betygspoäng, och en **RASE på 1,25** — faktoriseringsmaskinen förklarar ungefär hälften
av variansen i ett nöjdhetsbetyg på 1-10 som har en standardavvikelse på 1,81, så den lär
sig verkligen struktur i stället för att predikera det övergripande medelvärdet.
Residualsammanfattningen bekräftar detta: residualmedelvärdet är i praktiken noll (0,020)
och de predikterade betygen spänner från 5,80 till 9,54 (standardavvikelse 1,27), vilket
följer — även om det inte helt matchar — den observerade spridningen.

Tabellen för anläggning x specialitet gör den latenta modellen till något ett
vårdkvalitetsteam kan agera på. Modellen rankar `Centralkliniken`/`Kardiologi` högst
(predikterat 9,54) och `Sydkliniken`/`Kardiologi` lägst (predikterat 5,80), vilket
återskapar den inbäddade interaktionen där kardiologi är utmärkt på vissa enheter och svag
på andra. Kolumnen för absolut fel är ärlig om modellens begränsningar: de två
onkologi-cellerna återskapas nästan exakt (medelabsolutfel 0,19-0,24), medan
`Nordkliniken`/`Kardiologi` underskattas (fel 2,33) eftersom dess sanna betyg hopar sig
vid det klippta 1-10-taket som en linjär faktoriseringsmaskin inte kan nå.

**Nästa steg** en praktiker kan ta: lägg till en `PARTITION`-hålldel för att skydda mot
överanpassning, justera `LEARN=` och `L2=` för att balansera bias mot varians, eller
utöka funktionsuppsättningen (vårdgivare, besökstyp, säsong) så att faktoriseringsmaskinen
kan modellera drivkrafter av högre ordning för upplevelsen. På en fullt licensierad
installation skulle ett större anläggning x specialitet-rutnät med fler möten per cell
låta modellen lösa upp finare interaktionsstruktur än den sexcellsdesign som visas här.